# Mixed-Layer Heat Budget Code

By Shay Magahey & lots of debugging help from Sam Stevenson
For identifying trends in the mixed layer heat budget and how they contribute ot changes in ENSO
Corrected code: 07/02/2024

Ref paper: Effectiveness of the Bjerknes stability index in representing ocean dynamics Graham 2014

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from advection_rd_ml import advection_rd_ml
from vertical_advection_rd_ml import vertical_advection_rd_ml

In [2]:
# note for checking: t_mld is temperature at base of ML, not integrated ML temp. do sel(MLD=15),THEN Tsub
# but sam's notes say T is either MLD avg or value below MLD?

## Define Necessary Helper Functions

In [3]:
# Average a variable above the time-varying mixed layer depth
def mld_avg_var(mld,var,z):
    zmask= (z < mld) # z above mixed layer
    masked_var = var.where(zmask, drop=True)
    print("masked_var shape:", masked_var.shape)
    var=var.convert_calendar("proleptic_gregorian",use_cftime=False)
    intfield=var.where(zmask, drop=True).mean('lev').squeeze()
    #print(intfield)
    intfield=xr.where(intfield!=0,intfield,np.nan)
    return intfield

In [4]:
# Extract the value of a variable immediately below the time-varying mixed layer depth
## NOTE: this may still need debugging since it returns just the maximum temperature, not the shallowest temperature
def sub_ML_avg_var(mld,var,z):
    zmask= (z > mld) # z below mixed layer, greater than bc counts down from 0 (0,5,15,25,etc)
    #print(zmask)
    var=var.convert_calendar("proleptic_gregorian",use_cftime=False)
    fld=var.where(zmask, drop=False).squeeze() # where zmask = true: zmask is greater than MLD
    #print(fld)
    i=(fld.notnull().argmax(dim='lev'))  
    #print(fld.lev[i]) giving warmest, not shallowest. Pick highest depth value that's below mixed layer
    fldsub=fld.isel(lev=i) # selecting the field at the bottom of the mixed layer
    fldsub=xr.where(fldsub!=0,fldsub,np.nan)
    return fldsub

In [5]:
# Calculate shortwave flux penetrating the mixed layer
# (assumes MLD in m)
# equations are from
def calc_qpen(SW,MLD):
    eu=2.71818
    #MLD=np.array(MLD,dtype=np.float128)
    T1=0.58*np.exp((-MLD)/(.35)) # crazy numbers
    T2=0.42*np.exp((-MLD)/(23))
    qpen=SW*(T1+T2)
    return qpen,T1,T2

In [6]:
# Sanity check Q_pen: "manual" calculation
288.600 * (0.58 * np.exp(-14.710/0.35) + 0.42 * np.exp(-14.710/23))

np.float64(63.94196411428062)

## Read in Data, Define Necessary Parameters

NOTE: this assumes that ensemble-member information has already been read in and processed by another function


make_netcdf_e3smv1LE.ipynb

In [7]:
# Time parameters
timesfut=["2070-01","2099-12"]
timespi=["1850-01","1880-01"]
tipast=["1850-01","2015-01"]
tissp=["2015-01","2100-01"]

# Regional averaging parameters
n3=[-5,5,210,270] #Nino3
n34=[-5,5,190,240] # nino3.4
n4=[-5,5,160,210]
pac=[-5,5,120,280]
regbox=[-40,40,90,300]

In [8]:
period='2070_2100'

In [9]:
# Open data files
ds_sst=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files/tos_1850_2100_E3SMv1LE.nc")
dsThist=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files/thetao_"+period+"_E3SMv1LE.nc")
dsWhist=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files/wo_"+period+"_E3SMv1LE.nc")
dsUhist=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files/uo_"+period+"_E3SMv1LE.nc")
dsVhist=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files/vo_"+period+"_E3SMv1LE.nc")
dsHF=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files//hfds_1850_2100_E3SMv1LE.nc")
dsSW=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files/rsds_1850_2100_E3SMv1LE.nc")
dsML=xr.open_dataset("/glade/work/smagahey/UCSB/codes/generated_files/mlotst_1850_2100_E3SMv1LE.nc")


In [10]:
# Restrict all data to the pre-industrial time period (timespi) and Pacific region (pac)

Thi=dsThist['thetao'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3])) 
Whi=dsWhist['wo'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3])) 
Uhi=dsUhist['uo'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3])) 
Vhi=dsVhist['vo'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3]))
HFi=dsHF['hfds'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3]),time=slice(timesfut[0],timesfut[1]))
SWi=dsSW['rsds'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3]),time=slice(timesfut[0],timesfut[1]))
MLDi=dsML['mlotst'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3]),time=slice(timesfut[0],timesfut[1]))
#sali=dsS['sos'].sel(lat=slice(regbox[0],regbox[1]),lon=slice(regbox[2],regbox[3]),time=slice(timesfut[0],timesfut[1]))


In [11]:
dsML

<xarray.Dataset> Size: 12GB
Dimensions:  (lat: 180, lon: 360, time: 3012, member: 16)
Coordinates:
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * time     (time) object 24kB 1850-01-31 00:00:00 ... 2100-12-31 00:00:00
Dimensions without coordinates: member
Data variables:
    mlotst   (member, time, lat, lon) float32 12GB ...

In [12]:
# Convert calendar types for all data
Thi=Thi.convert_calendar('proleptic_gregorian',use_cftime=False)
Whi=Whi.convert_calendar('proleptic_gregorian',use_cftime=False)
Uhi=Uhi.convert_calendar('proleptic_gregorian',use_cftime=False)
Vhi=Vhi.convert_calendar('proleptic_gregorian',use_cftime=False)
SWi=SWi.convert_calendar('proleptic_gregorian',use_cftime=False)
HFi=HFi.convert_calendar('proleptic_gregorian',use_cftime=False)
MLDi=MLDi.convert_calendar('proleptic_gregorian',use_cftime=False)
#sali=sali.convert_calendar('proleptic_gregorian',use_cftime=False)

In [13]:
'''
Thi=Thi-Thi.mean('member')
Whi=Whi-Whi.mean('member')
Uhi=Uhi-Uhi.mean('member')
Vhi=Vhi-Vhi.mean('member')
SWi=SWi-SWi.mean('member')
HFi=HFi-HFi.mean('member')
MLDi=MLDi-MLDi.mean('member')
'''

"\nThi=Thi-Thi.mean('member')\nWhi=Whi-Whi.mean('member')\nUhi=Uhi-Uhi.mean('member')\nVhi=Vhi-Vhi.mean('member')\nSWi=SWi-SWi.mean('member')\nHFi=HFi-HFi.mean('member')\nMLDi=MLDi-MLDi.mean('member')\n"

In [14]:
# Calculate mixed-layer averaged quantities
MLDi=MLDi.isel(member=slice(0,15))
int_U=mld_avg_var(MLDi,Uhi,Uhi.lev)
int_V=mld_avg_var(MLDi,Vhi,Vhi.lev)
int_W=mld_avg_var(MLDi,Whi,Whi.lev)
int_T=mld_avg_var(MLDi,Thi,Thi.lev)

masked_var shape: (15, 360, 30, 80, 210)
masked_var shape: (15, 360, 30, 80, 210)
masked_var shape: (15, 360, 30, 80, 210)
masked_var shape: (15, 360, 30, 80, 210)


In [15]:
# Get the temperature and zonal, meridional, vertical velocity immediately below the mixed layer
sub_int_T=sub_ML_avg_var(MLDi,Thi,Thi.lev)
sub_int_U=sub_ML_avg_var(MLDi,Uhi,Uhi.lev)
sub_int_V=sub_ML_avg_var(MLDi,Vhi,Vhi.lev)
sub_int_W=sub_ML_avg_var(MLDi,Whi,Whi.lev)


In [16]:
MLDi_anom=MLDi.groupby('time.month')-MLDi.groupby('time.month').mean()
SWi_anom=SWi.groupby('time.month')-SWi.groupby('time.month').mean()

## Calculate Surface Flux Term

In [17]:
# Penetrative shortwave flux
# is SWi the correct input SW?
# is HFLX the correct input surface heat flux?

Q_pen,T1,T2=calc_qpen(SWi,MLDi) # NOT anomaly

# check heat flux

In [18]:
Q_pen_anom=Q_pen.groupby('time.month')-Q_pen.groupby('time.month').mean()
HFi_anom=HFi.groupby('time.month')-HFi.groupby('time.month').mean()


In [19]:
# calculate the difference then the anomaly

In [20]:
THFLx=HFi.isel(member=slice(0,15))-Q_pen # total heat flux: surface net minus penetrative
Th=THFLx/(3850*1025*MLDi) 
TH=Th.groupby('time.month')-Th.groupby('time.month').mean()

In [21]:

# range is not as bad for inputs as anomalies. results get messed if I include MLDi_anom in unit conv

## Calculate Horizontal Advective Terms

In [22]:
# Call function to compute Reynolds-decomposed horizontal advective terms
dTpdx,dTpdy,mnadv_climU,mnadv_climV,UpdTmdx,VpdTmdy,UmdTpdx,VmdTpdy,eddyU,eddyV,mneddyU,mneddyV,dTpdt,UmUp,VmVp,dTmdx,dTmdy,Um,Vm=advection_rd_ml(int_T,int_T.lat,int_T.lon,int_T.time,int_U,int_V)


Um= <xarray.DataArray 'uo' ()> Size: 4B
array(-0.03741102, dtype=float32)
Coordinates:
    lat      float64 8B -34.5
    lon      float64 8B 150.5
    month    int64 8B 5
dTpdx= <xarray.DataArray ()> Size: 8B
array(-1.07954966e-06)
Coordinates:
    lat      float64 8B -34.5
    lon      float64 8B 151.5
    time     datetime64[ns] 8B 2078-05-31
    month    int64 8B 5


## Calculate Vertical Advective Terms

In [23]:
dH,wsgn,wm,wentr,wtmp,tmp,up,thermo,eddy,updTpbar,dTm,dHdX,dHdY,Tm_mld,Tm_sub,tmpint,tmpsub,dTpdz,dTmdz=vertical_advection_rd_ml(MLDi,int_T.lat,int_T.lon,int_T,int_U,int_V,sub_int_W,sub_int_T)
# int_T is MLD average temperature
# June 2025 we want some intermediates:
# wm, dT'/dz (for thermocline), vm, dTpdy, 

<xarray.DataArray (member: 15, time: 360, lat: 80, lon: 209)> Size: 722MB
array([[[[ 1.00649017e-06,  3.54325577e-06,  4.29913375e-06, ...,
                      nan,  4.43312223e-07, -1.25996799e-05],
         [ 5.25631787e-06,  4.18318664e-06,  1.89566449e-06, ...,
                      nan,             nan, -1.42382829e-05],
         [ 2.76982382e-06,  1.29005612e-06, -1.49538263e-06, ...,
                      nan,             nan,             nan],
         ...,
         [            nan,             nan,             nan, ...,
          -4.31320884e-05, -2.76413939e-05, -1.74852580e-06],
         [            nan,             nan,             nan, ...,
          -3.82022898e-05, -3.90367114e-05,  2.75320615e-05],
         [            nan,             nan,             nan, ...,
          -5.31868081e-05, -1.18772680e-04, -5.20623174e-05]],

        [[-1.13920738e-05, -1.09225146e-05, -7.46521662e-06, ...,
                      nan,  8.85336522e-08, -1.11967757e-05],
         [-7.0

In [24]:
tmdiff=Tm_mld.sel(lon=slice(n34[2],n34[3])).mean('lat').mean('lon').isel(member=0)-Tm_sub.sel(lon=slice(n34[2],n34[3])).mean('lat').mean('lon').isel(member=0)



In [25]:
upwelling=up.drop_vars('month').drop_vars('lev')*86400*30.41
thermocline=thermo.drop_vars('month').drop_vars('lev')*86400*30.41
Qp=TH*86400*30.41
eddyvert=eddy.drop_vars('month').drop_vars('lev')*86400*30.41
updT=updTpbar*86400*30.41
ekU=UpdTmdx.drop_vars('month')*86400*30.41
ekV=VpdTmdy.drop_vars('month')*86400*30.41
T11=UmdTpdx*86400*30.41
T12=VmdTpdy*86400*30.41
eddyU=eddyU.drop_vars('month')*86400*30.41
eddyV=eddyV.drop_vars('month')*86400*30.41
dTpdt=dTpdt.drop_vars('month')*86400*30.41
mneddyU=mneddyU*86400*30.41
mneddyV=mneddyV*86400*30.41
# conversions seem fine

In [26]:
#temptend = Qpo - T1 - ekman - eddyH - upwelling - thermocline - eddyV


In [27]:
#hb_data=xr.Dataset({'Qp':Qp.astype(np.float32),'dTpdt':dTpdt,'UmdTpdx':T11,'VmdTpdy':T12,'eddyU':eddyU,'eddyV':eddyV,'ekU':ekU,'ekV':ekV,'upwelling':upwelling,'thermocline':thermocline,'eddyvert':eddyvert,'mneddyU':mneddyU,'mneddyV':mneddyV})
#hb_data.to_netcdf('/glade/work/smagahey/UCSB/codes/paper1/fut_corrQ_hb_terms.nc')


In [28]:
#wtmp

In [29]:
component_terms=xr.Dataset({'wm':wm, 'dTmdx':dTmdx,'dTmdy':dTmdy,'dTmdz':dTmdz,'dTpdx':dTpdx,'dTpdy':dTpdy,'dTpdz':dTpdz,'wp':wtmp.drop_vars('lev'),'Up':UmUp,'Vp':VmVp,'Um':Um,'Vm':Vm})



In [30]:
#Qp.sel(lon=slice(190,240)).mean('lat').mean('lon').mean('member').plot()

In [31]:
component_terms.to_netcdf('/glade/work/smagahey/UCSB/codes/paper1_new/paper1/hb_analysis/fut_allcomponent_terms.nc')


In [32]:
#therm_terms_hist=xr.Dataset({'Tm_mld':Tm_mld,'Tm_sub':Tm_sub,'Tp_mld':tmpint,'Tp_sub':tmpsub,'wm':wm})
#therm_terms_hist.to_netcdf('/glade/work/smagahey/UCSB/codes/paper1/therm_terms_hist.nc')